In [ ]:
import pandas as pd
import re
from pathlib import Path

# Đường dẫn tới file dữ liệu gốc
DATA_PATH = '../../Mail-Spam/datasets/raw/email_dataset_github.csv'
df = pd.read_csv(DATA_PATH)

# 1. Kiểm tra cân bằng lớp
if 'isSpam' in df.columns:
    class_counts = df['isSpam'].value_counts()
    print('Số lượng mỗi lớp:')
    print(class_counts)
    print('Tỉ lệ mỗi lớp:')
    print(class_counts / len(df))
else:
    print('Không tìm thấy cột isSpam để kiểm tra cân bằng lớp.')

# 2. Kiểm tra số từ trên mỗi email
if 'msg' in df.columns:
    df['num_words'] = df['msg'].apply(lambda x: len(str(x).split()))
    print('\nSố từ trên mỗi email:')
    print('Min:', df['num_words'].min())
    print('Max:', df['num_words'].max())
    print('Mean:', df['num_words'].mean())
else:
    print('Không tìm thấy cột msg để kiểm tra số từ.')

# 3. Kiểm tra dữ liệu bẩn
print('\nKiểm tra dữ liệu bẩn:')
if 'msg' in df.columns:
    empty_msgs = df['msg'].isnull().sum() + (df['msg'].astype(str).str.strip() == '').sum()
    print(f'Số tin nhắn rỗng: {empty_msgs}')
    special_char_msgs = df['msg'].apply(lambda x: bool(re.fullmatch(r'[^\w\s]+', str(x))))
    print(f'Số tin nhắn chỉ chứa ký tự đặc biệt: {special_char_msgs.sum()}')
else:
    print('Không tìm thấy cột msg để kiểm tra dữ liệu bẩn.')

# 4. Loại trùng theo nội dung đã chuẩn hoá + label và lưu dataset mới
if {'msg', 'isSpam'}.issubset(df.columns):
    def normalize_text(text):
        text = str(text).lower()
        text = re.sub(r'<.*?>', ' ', text)
        text = re.sub(r'http[s]?://\S+', ' ', text)
        text = re.sub(r'!{2,}', ' exclaim ', text)
        text = re.sub(r'\${2,}', ' money_symbol ', text)
        text = re.sub(r'\b\d{10,}\b', ' phone_number ', text)
        text = re.sub(r'[^\w\s]', ' ', text)
        text = re.sub(r'\s+', ' ', text).strip()
        return text

    total_before = len(df)
    df['_norm_msg'] = df['msg'].apply(normalize_text)

    duplicate_norm_count = df.duplicated(subset=['_norm_msg', 'isSpam']).sum()
    conflict_norms = df.groupby('_norm_msg')['isSpam'].nunique()
    conflict_norms = conflict_norms[conflict_norms > 1].index.tolist()

    print('\nKiểm tra trùng lặp sau chuẩn hoá:')
    print(f'Số dòng trùng theo nội dung đã chuẩn hoá + label: {duplicate_norm_count}')

    if conflict_norms:
        print('CẢNH BÁO: Có nội dung đã chuẩn hoá giống nhau nhưng label khác nhau:')
        conflict_view = df[df['_norm_msg'].isin(conflict_norms)][['msg', 'isSpam', '_norm_msg']].drop_duplicates().sort_values(['_norm_msg', 'isSpam'])
        print(conflict_view.head(50).to_string(index=False))
    else:
        print('Không phát hiện nội dung đã chuẩn hoá giống nhau nhưng label khác nhau.')

    df = df.drop_duplicates(subset=['_norm_msg', 'isSpam'], keep='first').reset_index(drop=True)
    df = df.drop(columns=['_norm_msg'])

    processed_path = Path(r'D:\MyData\Disk D\HocKi6\DL\Rain-Forecast\datasets\processed\email_dataset_github_processed.csv')
    processed_path.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(processed_path, index=False, encoding='utf-8-sig')

    print(f'Đã xoá các dòng trùng theo nội dung đã chuẩn hoá + label.')
    print(f'Tổng số dòng trước khi xoá: {total_before}')
    print(f'Tổng số dòng sau khi xoá: {len(df)}')
    print(f'Đã lưu dataset đã xử lý vào: {processed_path}')
else:
    print('Không đủ cột msg/isSpam để kiểm tra trùng.')

Shape của dataset gốc: (499999, 2)
Shape sau khi loại bỏ duplicates: (273722, 2)
Đã lưu dataset processed tại: d:\MyData\Disk D\HocKi6\DL\Rain-Forecast\datasets\processed\email_dataset_github_processed.csv
